In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
os.chdir('/kaggle/input/data-science-london-scikit-learn')

In [ ]:
labels = pd.read_csv('trainLabels.csv', header=None)

In [ ]:
df = pd.read_csv('train.csv', header=None)
df.head(2)

In [ ]:
test = pd.read_csv('test.csv', header=None)
test.head(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
full_df = pd.concat([df, labels], axis=1)
full_df.head(3)

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, classification_report, auc
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()
X_train, X_test, Y_train, Y_test = train_test_split(df, labels, test_size = 0.2)

In [ ]:
X_train = scaler.fit_transform(X_train)

In [ ]:
X_test = scaler.transform(X_test)

In [ ]:
full_df.columns

In [ ]:
full_df.describe()

In [ ]:
sorted(df.corr()[labels.columns[0]])

In [ ]:
import xgboost as xgb

In [ ]:
xg = xgb.XGBClassifier()
xg.fit(X_train, Y_train)

In [ ]:
preds = xg.predict(X_test)
print("Classification Report:\n", classification_report(Y_test, preds))

In [ ]:
test.head()

In [ ]:
def plot_roc_auc_curve(Y_test, preds):
    fpr, tpr, thresh = roc_curve(Y_test, preds)
    roc_auc = auc(fpr, tpr)
    plt.figure()  
    plt.plot(fpr, tpr, label='ROC curve (area = %0.2f)' % roc_auc)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.show()
    print(roc_auc_score(Y_test, preds))

In [ ]:
plot_roc_auc_curve(Y_test, preds)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier()
rf.fit(X_train, Y_train)

In [ ]:
rf_preds = rf.predict(X_test)

In [ ]:
plot_roc_auc_curve(Y_test, rf_preds)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression()
lr.fit(X_train, Y_train)

In [ ]:
lr_preds = lr.predict(X_test)
plot_roc_auc_curve(Y_test, lr_preds)

In [ ]:
from sklearn.neighbors import KNeighborsClassifier 
knn = KNeighborsClassifier(10)
knn.fit(X_train, Y_train)

In [ ]:
knn_preds = knn.predict(X_test)
plot_roc_auc_curve(Y_test, knn_preds)

In [ ]:
test_preds = knn.predict(test)

In [ ]:
sub = pd.DataFrame({
    "Id": range(1,1+len(test_preds)), 
    "Solution": test_preds        
})
sub.head()

In [ ]:
sub.to_csv('/kaggle/working/predictions.csv', index=False)